In [45]:
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [46]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

72


In [12]:
import json

with open('documents.json', 'rt') as f_in:
    documents_raw = json.load(f_in)

documents = []
for course in documents_raw:
    course_name = course['course']
    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

print(len(documents))

948


In [13]:
## ## 2


In [22]:
import json
print(json.dumps(documents[0], indent=2))

{
  "content": "# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we'll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type \"how are\" in WhatsApp, it suggests\n\"you\" as the next word. \"How are you\" is the most common continuation.\nYour phone use

In [23]:

from minsearch import Index

index = Index(
    text_fields=["content", "filename"],
    keyword_fields=["course"]
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [24]:
## 3

In [25]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-13 21:03:29--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8001::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-13 21:03:30 (6.83 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [27]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

client = OpenAI()

def search(query, num_results=5):
    return index.search(
        query,
        boost_dict={'question': 3.0, 'section': 0.5},
        num_results=num_results
    )

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["filename"])
        lines.append(doc["content"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(query, search_results):
    context = build_context(search_results)
    return f"""QUESTION: {query}

CONTEXT:
{context}""".strip()

def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': prompt}
        ]
    )
    answer = response.choices[0].message.content
    usage = response.usage
    return answer, usage

def rag(query):
    results = search(query)
    prompt = build_prompt(query, results)
    answer, usage = llm(prompt)
    return answer, usage

In [28]:

answer, usage = rag(question)

print("Input tokens:", usage.prompt_tokens)
print("Output tokens:", usage.completion_tokens)
print("Total tokens:", usage.total_tokens)

Input tokens: 7016
Output tokens: 281
Total tokens: 7297


In [29]:
###  Q4. Chunking
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [30]:
### Q5. RAG with chunking

from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

question = "How does the agentic loop keep calling the model until it stops?"

import os
#os.environ["OPENAI_API_KEY"] = "sk-proj-yourkey"
from openai import OpenAI
client = OpenAI()

def search(query, num_results=5):
    return chunk_index.search(
        query,
        num_results=num_results
    )

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["filename"])
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(query, search_results):
    context = build_context(search_results)
    return f"""QUESTION: {query}\n\nCONTEXT:\n{context}""".strip()

def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': prompt}
        ]
    )
    return response.choices[0].message.content, response.usage

def rag(query):
    results = search(query)
    prompt = build_prompt(query, results)
    return llm(prompt)

answer, usage = rag(question)
print("Answer:", answer)
print("Input tokens:", usage.prompt_tokens)


Answer: The agentic loop continues to call the model until it receives a response that has no function calls. The loop operates using a `while` statement, and it checks for function calls in the model's response during each iteration. The loop will stop when the model returns an answer without requesting any further tool calls, indicated by the `has_function_calls` flag being `False`. 

In summary, the exit condition for the loop is simple: if there are no function calls in the current response from the model, the loop breaks. Additional safety measures can be implemented, such as limiting the maximum number of iterations, but the core mechanism remains centered around this flag.
Input tokens: 2295


In [31]:
answer_q3, usage_q3 = rag(question)

answer_q5, usage_q5 = rag(question)
print("Input tokens:", usage_q5.prompt_tokens)

# Compare
ratio = usage_q3.prompt_tokens / usage_q5.prompt_tokens
print(f"Q3 tokens: {usage_q3.prompt_tokens}")
print(f"Q5 tokens: {usage_q5.prompt_tokens}")
print(f"Ratio: {ratio:.1f}x fewer")

Input tokens: 2295
Q3 tokens: 2295
Q5 tokens: 2295
Ratio: 1.0x fewer


In [44]:


call_count = 0

def search(query: str) -> list:
    """Search the course materials for relevant information about the given query."""
    global call_count
    call_count += 1
    print(f"Search #{call_count}: {query}")
    return chunk_index.search(query, num_results=3)

tools_schema = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course materials for relevant information about the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    }
}]

messages = [
    {"role": "system", "content": """You're a course teaching assistant. Answer the student's question 
using the search tool. Make multiple searches with different keywords before answering."""},
    {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}
]

while True:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools_schema
    )
    
    msg = response.choices[0].message
    messages.append(msg)
    
    if not msg.tool_calls:
        break
    
    for tool_call in msg.tool_calls:
        query = json.loads(tool_call.function.arguments)["query"]
        result = search(query)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

print("\nAnswer:", msg.content)
print(f"\nTotal search calls: {call_count}")

Search #1: agentic loop
Search #2: plain RAG

Answer: The **agentic loop** and **plain RAG (Retrieval Augmented Generation)** are two concepts used in AI workflows, but they operate differently.

### Agentic Loop

The **agentic loop** involves a dynamic workflow where an AI agent can make decisions about the sequence of actions it takes based on the current context and goals. The agent continuously interacts with the environment, retrieves information, and acts upon it until a final result is achieved. This is often achieved through a loop structure that repeatedly calls a language model (LLM) while leveraging previous interactions to inform future actions.

In practical applications, frameworks manage this loop automatically. For example, in Kestra, you define the agent's goals and the tools it can use, while the framework takes care of managing the loop and conversation history. This allows the agent to adapt its strategy based on real-time data and interactions, making it suitable f

In [42]:
import inspect
from toyaikit.chat import ChatAssistant
print(inspect.signature(ChatAssistant.__init__))

(self, tools: toyaikit.tools.Tools, developer_prompt: str, chat_interface: toyaikit.chat.interface.ChatInterface, llm_client: toyaikit.llm.LLMClient)
